In [ ]:
import subprocess
import os

LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imágenes: {n_images} | Labels: {n_labels}")


In [ ]:
import glob
import numpy as np

def leer_cajas_reales(label_path):
    """Lee un .txt de labels y devuelve una lista de (xc, yc, w, h)"""
    cajas = []
    if not os.path.exists(label_path):
        return cajas
    with open(label_path) as f:
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            xc, yc, w, h = map(float, partes[1:5])
            cajas.append((xc, yc, w, h))
    return cajas

# Cada imagen predice un número FIJO de "slots" de caja (MAX_BOXES, igual que
# antes), pero ahora cada slot es solo (objectness, xc, yc, w, h): sin clase.
max_danos_en_imagen = 0
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    n = len(leer_cajas_reales(label_path))
    max_danos_en_imagen = max(max_danos_en_imagen, n)

MAX_BOXES = max_danos_en_imagen + 5
print(f"Máximo de daños anotados en una sola imagen: {max_danos_en_imagen}")
print(f"MAX_BOXES elegido: {MAX_BOXES}")


In [ ]:
def parse_label_to_boxes(label_path, max_boxes=MAX_BOXES):
    """Target de tamaño FIJO (max_boxes, 5): columna 0 = objectness
    (1 = hay daño real en este slot, 0 = padding), columnas 1-4 = caja
    (xc, yc, w, h) en coordenadas absolutas de la imagen completa (0-1)."""
    target = np.zeros((max_boxes, 5), dtype=np.float32)
    cajas = leer_cajas_reales(label_path)[:max_boxes]
    for i, (xc, yc, w, h) in enumerate(cajas):
        target[i, 0] = 1.0
        target[i, 1:5] = [xc, yc, w, h]
    return target


### PARA PROBAR

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def plot_real_box_on_image(label_path, ax):
    """Dibuja las cajas reales anotadas, directo en coordenadas absolutas de
    la imagen (sin celdas de por medio)."""
    boxes = parse_label_to_boxes(label_path)
    valid = boxes[:, 0] == 1

    img_name = os.path.basename(label_path).replace(".txt", "")
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break

    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)

    for xc, yc, w, h in boxes[valid, 1:]:
        box_w = w * img_w
        box_h = h * img_h
        x0 = xc * img_w - box_w / 2
        y0 = yc * img_h - box_h / 2

        rect = patches.Rectangle((x0, y0), box_w, box_h,
                                  linewidth=2, edgecolor="red",
                                  facecolor="none")
        ax.add_patch(rect)

    ax.set_title(img_name, fontsize=9)
    ax.axis("off")


# Elegimos 2 labels random que tengan al menos un daño anotado
labels_con_dano = []
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    if len(leer_cajas_reales(label_path)) > 0:
        labels_con_dano.append(label_path)

muestras = random.sample(labels_con_dano, 2)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, label_path in zip(axes, muestras):
    plot_real_box_on_image(label_path, ax)

plt.tight_layout()
plt.show()


In [ ]:
image_files = sorted(glob.glob(f"{LOCAL_DATA_DIR}/images/*.jpg"))

def get_label_path(image_path):
    return image_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

# Split 80/20. Sin sobremuestreo por clase (ya no hay clases acá): la única
# forma de desbalance posible es "imagen con daño" vs "sin daño", y eso lo
# maneja el término de objectness en la loss (más abajo).
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
random.seed(RANDOM_SEED)
image_files_shuffled = list(image_files)
random.shuffle(image_files_shuffled)
n_train = int(len(image_files_shuffled) * TRAIN_RATIO)
train_paths = image_files_shuffled[:n_train]
val_paths = image_files_shuffled[n_train:]

print(f"Imágenes: {len(image_files)} | Train: {len(train_paths)} | Validation: {len(val_paths)}")


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-4
CHECKPOINT_PATH = "../models/best_locator.keras"
RUN_TRAINING = True
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def load_sample(image_path):
    image_path = image_path.numpy().decode("utf-8")
    label_path = get_label_path(image_path)
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    target = parse_label_to_boxes(label_path)
    return img_array, target

def tf_load_sample(image_path):
    img, target = tf.py_function(load_sample, [image_path], [tf.float32, tf.float32])
    img.set_shape([224, 224, 3])
    target.set_shape([MAX_BOXES, 5])
    return img, target

train_ds = tf.data.Dataset.from_tensor_slices(train_paths).shuffle(1000).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(val_paths).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Datasets armados")
print("GPUs detectadas:", tf.config.list_physical_devices("GPU"))


In [ ]:
sample_images, sample_targets = next(iter(train_ds))
frac_positivas = (sample_targets[..., 0] == 1).numpy().mean()
print(f"Fracción de slots con daño real en un batch: {frac_positivas:.4f}")


In [ ]:
# Reutilizamos el modelo 0 (best_model.keras, daño/no-daño) como extractor
# de features. Nos quedamos con todo lo anterior a GlobalAveragePooling.
modelo0 = tf.keras.models.load_model("../models/best_model.keras")

backbone_output = None
for layer in modelo0.layers:
    if isinstance(layer, tf.keras.layers.MaxPooling2D):
        backbone_output = layer.output
assert backbone_output is not None, "No se encontró ninguna capa MaxPooling2D en modelo0"

backbone = tf.keras.Model(inputs=modelo0.input, outputs=backbone_output, name="backbone_reutilizado")
backbone.trainable = False  # se descongela en la Fase 2

print("Backbone reutilizado, forma de salida:", backbone.output_shape)


In [ ]:
from tensorflow.keras import layers

# Cabeza de localización: sin celdas, MAX_BOXES slots de (objectness, caja).
# Sin cabeza de clase.
x = backbone.output  # (14, 14, 256)
x = layers.Conv2D(256, 3, padding="same", activation="relu", name="loc_conv2d")(x)
x = layers.GlobalAveragePooling2D(name="loc_gap")(x)
x = layers.Dense(256, activation="relu", name="loc_dense")(x)

bias_inicial_objectness = np.log(frac_positivas / (1 - frac_positivas))
out_obj = layers.Dense(
    MAX_BOXES, activation="linear", name="loc_obj",
    bias_initializer=tf.keras.initializers.Constant(bias_inicial_objectness),
)(x)
out_obj = layers.Reshape((MAX_BOXES, 1))(out_obj)

out_box = layers.Dense(MAX_BOXES * 4, activation="sigmoid", name="loc_box")(x)
out_box = layers.Reshape((MAX_BOXES, 4))(out_box)

output = layers.Concatenate(axis=-1, name="loc_output")([out_obj, out_box])
modelo_localizacion = tf.keras.Model(inputs=backbone.input, outputs=output, name="modelo_localizacion")
modelo_localizacion.summary()


### Matching y loss

Igual que antes, no hay celdas: el matching Húngaro decide en cada batch qué
slot predicho le corresponde a cada daño real, pero ahora el costo de esa
asignación es **solo la distancia entre cajas** (no hay clase que comparar).
La loss final es objectness (sigmoid cross-entropy) + error de caja, sin
ningún peso de clase multiplicando nada.

In [ ]:
from scipy.optimize import linear_sum_assignment

def hungarian_match_batch(true_mask, true_boxes, pred_boxes):
    """Por imagen del batch, asigna cada daño real al slot predicho con la
    caja más parecida. Devuelve, por slot, el índice del daño real que le
    tocó (-1 si no le tocó ninguno)."""
    batch = true_mask.shape[0]
    match_idx = np.full((batch, true_boxes.shape[1]), -1, dtype=np.int32)
    for b in range(batch):
        valid = true_mask[b] > 0.5
        n_true = int(valid.sum())
        if n_true == 0:
            continue
        tb = true_boxes[b, valid]
        pb = pred_boxes[b]
        cost = np.abs(pb[:, None, :] - tb[None, :, :]).sum(axis=-1)
        pred_idx, true_idx = linear_sum_assignment(cost)
        real_idx = np.where(valid)[0]
        for p_i, t_i in zip(pred_idx, true_idx):
            match_idx[b, p_i] = real_idx[t_i]
    return match_idx

def detection_loss(y_true, y_pred, lambda_coord=5.0):
    obj_logits = y_pred[..., 0]
    box_pred = y_pred[..., 1:5]

    match_idx = tf.numpy_function(
        hungarian_match_batch,
        [y_true[..., 0], y_true[..., 1:5], box_pred],
        tf.int32,
    )
    match_idx.set_shape([None, MAX_BOXES])

    obj_target = tf.cast(match_idx >= 0, tf.float32)
    loss_obj = tf.nn.sigmoid_cross_entropy_with_logits(labels=obj_target, logits=obj_logits)

    safe_idx = tf.maximum(match_idx, 0)
    matched_box = tf.gather(y_true[..., 1:5], safe_idx, batch_dims=1)
    loss_box = tf.reduce_sum(tf.square(matched_box - box_pred), axis=-1)
    loss_box = obj_target * loss_box

    return tf.reduce_mean(loss_obj) + lambda_coord * tf.reduce_mean(loss_box)

# clipnorm: si algún batch produce un gradiente enorme, se recorta antes de
# aplicarlo — red de seguridad contra la explosión numérica que tuvimos antes.
modelo_localizacion.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=detection_loss,
)


In [ ]:
expected_output_shape = tuple(modelo_localizacion.output_shape[1:])
actual_target_shape = tuple(sample_targets.shape[1:])
print("Batch imágenes:", sample_images.shape)
print("Batch targets:", sample_targets.shape)
print("Salida del modelo:", modelo_localizacion.output_shape)

assert actual_target_shape == expected_output_shape, (
    f"Targets incompatibles con la salida del modelo: {actual_target_shape} != {expected_output_shape}"
)

sample_predictions = modelo_localizacion(sample_images[:1], training=False)
assert tuple(sample_predictions.shape[1:]) == expected_output_shape
print("Verificación OK.")


In [ ]:
def crear_callbacks():
    return [
        tf.keras.callbacks.ModelCheckpoint(
            CHECKPOINT_PATH, monitor="val_loss", mode="min", save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1,
        ),
        # Corta el entrenamiento apenas aparece un NaN, en vez de seguir
        # 50 épocas entrenando sobre pesos ya rotos.
        tf.keras.callbacks.TerminateOnNaN(),
    ]

print(f"Callbacks preparados. El mejor localizador se guardará en: {CHECKPOINT_PATH}")


### Entrenamiento

Dos fases: warm-up con el backbone congelado, después fine-tuning completo
con LR más bajo. Solo corre si `RUN_TRAINING = True`.

In [ ]:
if not RUN_TRAINING:
    print("Entrenamiento preparado pero no iniciado. Cambiar RUN_TRAINING = True para ejecutar model.fit().")
else:
    print("=== Fase 1: warm-up de la cabeza (backbone congelado) ===")
    history_warmup = modelo_localizacion.fit(
        train_ds, validation_data=val_ds, epochs=5, callbacks=crear_callbacks(), verbose=1,
    )

    print("=== Fase 2: fine-tuning completo (backbone descongelado) ===")
    backbone.trainable = True
    modelo_localizacion.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10, clipnorm=1.0),
        loss=detection_loss,
    )
    history = modelo_localizacion.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=crear_callbacks(), verbose=1,
    )


In [ ]:
if "history" not in globals():
    print("Todavía no hay history: ejecutar primero el entrenamiento.")
else:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history.get("loss", []), label="training loss")
    plt.plot(history.history.get("val_loss", []), label="validation loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title("Evolución de la loss del localizador (fase 2)")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    best_locator = tf.keras.models.load_model(
        CHECKPOINT_PATH, custom_objects={"detection_loss": detection_loss},
    )
    print(f"Mejor localizador cargado desde: {CHECKPOINT_PATH}")
else:
    best_locator = modelo_localizacion
    print(f"Aún no existe checkpoint en {CHECKPOINT_PATH}; usando el modelo en memoria.")


### Predicción

En inferencia varios slots pueden disparar para el mismo daño: NMS se queda
con una sola caja por daño. Como acá hay una sola clase, NMS es más simple
que en los intentos anteriores (no hace falta agrupar por clase).

In [ ]:
def calcular_iou(box1, box2):
    """box format: (xc, yc, w, h) normalizado 0-1."""
    x1_min, y1_min = box1[0] - box1[2] / 2, box1[1] - box1[3] / 2
    x1_max, y1_max = box1[0] + box1[2] / 2, box1[1] + box1[3] / 2
    x2_min, y2_min = box2[0] - box2[2] / 2, box2[1] - box2[3] / 2
    x2_max, y2_max = box2[0] + box2[2] / 2, box2[1] + box2[3] / 2

    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def non_max_suppression(predicciones, umbral_iou_nms=0.4):
    """predicciones: lista de (xc, yc, w, h, objectness). Una sola clase,
    así que no hace falta agrupar nada antes de aplicar NMS."""
    predicciones = sorted(predicciones, key=lambda p: p[4], reverse=True)
    resultado = []
    while predicciones:
        mejor = predicciones.pop(0)
        resultado.append(mejor)
        predicciones = [p for p in predicciones if calcular_iou(mejor[:4], p[:4]) < umbral_iou_nms]
    return resultado


def predecir_cajas(modelo, image_path, umbral_objectness=0.3, umbral_iou_nms=0.4, img_size=IMG_SIZE):
    """Corre el modelo sobre una imagen y devuelve las cajas predichas
    (xc, yc, w, h, objectness), ya filtradas por NMS."""
    img = Image.open(image_path).convert("RGB").resize(img_size)
    img_array = tf.keras.utils.img_to_array(img)
    pred = modelo(tf.expand_dims(img_array, 0), training=False)[0]  # (MAX_BOXES, 5)

    predicciones = []
    for slot in range(pred.shape[0]):
        objectness = tf.sigmoid(pred[slot, 0]).numpy()
        if objectness < umbral_objectness:
            continue
        xc, yc, w, h = pred[slot, 1:5].numpy()
        predicciones.append((xc, yc, w, h, objectness))

    return non_max_suppression(predicciones, umbral_iou_nms)


def plot_pred_box_on_image(label_path, ax, umbral_objectness=0.3):
    """Dibuja las cajas PREDICHAS: mismo estilo e interfaz que
    plot_real_box_on_image, sin texto de clase (este modelo no la sabe)."""
    img_name = os.path.basename(label_path).replace(".txt", "")
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break

    cajas = predecir_cajas(best_locator, img_path, umbral_objectness=umbral_objectness)
    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)

    for xc, yc, w, h, objectness in cajas:
        box_w = w * img_w
        box_h = h * img_h
        x0 = xc * img_w - box_w / 2
        y0 = yc * img_h - box_h / 2

        rect = patches.Rectangle((x0, y0), box_w, box_h,
                                  linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x0, y0 - 4, f"{objectness:.2f}", color="red", fontsize=7, weight="bold")

    ax.set_title(f"Predicción: {img_name}", fontsize=9)
    ax.axis("off")


### Demo: label random con daño, real vs. predicho

In [ ]:
labels_val_con_dano = [
    get_label_path(p) for p in val_paths
    if len(leer_cajas_reales(get_label_path(p))) > 0
]
label_demo = random.choice(labels_val_con_dano)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_real_box_on_image(label_demo, axes[0])
axes[0].set_title(f"Real: {os.path.basename(label_demo)}", fontsize=10)
plot_pred_box_on_image(label_demo, axes[1], umbral_objectness=0.3)
plt.tight_layout()
plt.show()


### (Opcional) Evaluación cuantitativa

In [ ]:
def evaluar_localizacion(modelo, paths, umbral_objectness=0.5, umbral_iou=0.5):
    total_predicciones = total_reales = verdaderos_positivos = 0
    ious_matched = []

    for image_path in paths:
        predicciones = predecir_cajas(modelo, image_path, umbral_objectness=umbral_objectness)
        reales = leer_cajas_reales(get_label_path(image_path))

        total_predicciones += len(predicciones)
        total_reales += len(reales)

        usados = set()
        for pred_box in predicciones:
            mejor_iou, mejor_idx = 0, -1
            for idx, real_box in enumerate(reales):
                if idx in usados:
                    continue
                iou = calcular_iou(pred_box[:4], real_box[:4])
                if iou > mejor_iou:
                    mejor_iou, mejor_idx = iou, idx
            if mejor_iou >= umbral_iou:
                verdaderos_positivos += 1
                ious_matched.append(mejor_iou)
                usados.add(mejor_idx)

    precision = verdaderos_positivos / total_predicciones if total_predicciones else 0.0
    recall = verdaderos_positivos / total_reales if total_reales else 0.0
    iou_promedio = np.mean(ious_matched) if ious_matched else 0.0

    print(f"Umbral objectness={umbral_objectness} | Predicciones: {total_predicciones} | "
          f"Reales: {total_reales} | TP (IoU>={umbral_iou}): {verdaderos_positivos}")
    print(f"  Precision: {precision:.3f} | Recall: {recall:.3f} | IoU promedio: {iou_promedio:.3f}")
    return precision, recall, iou_promedio


for umbral in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    evaluar_localizacion(best_locator, val_paths, umbral_objectness=umbral)
